In [7]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import torchvision.transforms as transforms


sys.path.append(os.path.abspath(".."))
from scripts.Loading_Dataset import LiverDataset

from monai.networks.nets.swin_unetr import SwinUNETR


In [8]:
# =========================================
# DEVICE
# =========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
# =========================================
# DATASET () ONLY TENSOR CONVERSION INSIDE DATASET)
# =========================================
root_dir = "../Dataset"

transform = transforms.ToTensor()

dataset = LiverDataset(root_dir=root_dir, transform=transform)

dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [10]:
# =========================================
# MONAI SWIN UNETR MODEL
# =========================================
model = SwinUNETR(
    in_channels=3,
    out_channels=1,
    feature_size=48,
)

model = model.to(device)
model.eval()

SwinUNETR(
  (swinViT): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(3, 48, kernel_size=(2, 2, 2), stride=(2, 2, 2))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers1): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0-1): 2 x SwinTransformerBlock(
            (norm1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=48, out_features=144, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=48, out_features=48, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (mlp): MLPBlock(
              (linear1): Linear(in_features=48, out_features=192, bias=True)
              (linear2): Linear(in_feature

In [11]:
# =========================================
# FEATURE EXTRACTION
# =========================================
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(dataloader, desc="Extracting features"):

        # images should already be tensors from dataset
        images = images.to(device)

        # forward through SwinViT encoder
        features = model.swinViT(images)

        # deepest feature map
        features = features[-1]   # [B, C, H, W]

        # global average pooling
        features = torch.mean(features, dim=[2, 3])  # [B, C]

        all_features.append(features.cpu())
        all_labels.append(labels)

Extracting features:   0%|          | 0/112 [00:32<?, ?it/s]


RuntimeError: Given groups=1, weight of size [48, 3, 2, 2, 2], expected input[1, 32, 3, 224, 224] to have 3 channels, but got 32 channels instead

In [ ]:
# =========================================
# MERGE RESULTS
# =========================================
all_features = torch.cat(all_features, dim=0)
all_labels = torch.cat(all_labels, dim=0)

SwinUNETR(
  (swinViT): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(3, 48, kernel_size=(2, 2, 2), stride=(2, 2, 2))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers1): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0-1): 2 x SwinTransformerBlock(
            (norm1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=48, out_features=144, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=48, out_features=48, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (mlp): MLPBlock(
              (linear1): Linear(in_features=48, out_features=192, bias=True)
              (linear2): Linear(in_feature

In [ ]:
# =========================================
# OUTPUT
# =========================================
print("Feature shape:", all_features.shape)
print("Label shape:", all_labels.shape)

print("\nSample feature vector:")
print(all_features[0])

Extracting features:   0%|          | 0/112 [00:27<?, ?it/s]


RuntimeError: Given groups=1, weight of size [48, 3, 2, 2, 2], expected input[1, 32, 3, 224, 224] to have 3 channels, but got 32 channels instead